# Блок 3. Практика: Хранение данных

В этом notebook мы:
1. Изучим работу с PostgreSQL для хранения логов
2. Научимся профилировать запросы с EXPLAIN ANALYZE
3. Познакомимся с DuckDB и его возможностями
4. Сравним производительность разных подходов

## Подготовка

Убедитесь, что инфраструктура из Блока 1 запущена и в базе есть данные.

```bash
cd ../solutions/lesson01
docker compose up -d
```

Генератор должен проработать некоторое время, чтобы накопились данные для анализа.

In [ ]:
import psycopg2
import duckdb
import time
from datetime import datetime

In [ ]:
# Параметры подключения к PostgreSQL
PG_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "security_logs",
    "user": "analyst",
    "password": "security123"  # Замените на ваш пароль из .env
}

# Строка подключения для DuckDB
PG_CONNECTION_STRING = f"postgresql://{PG_CONFIG['user']}:{PG_CONFIG['password']}@{PG_CONFIG['host']}:{PG_CONFIG['port']}/{PG_CONFIG['database']}"

## Часть 1: PostgreSQL — изучаем схему и данные

In [ ]:
# Подключаемся к PostgreSQL
conn = psycopg2.connect(**PG_CONFIG)
cursor = conn.cursor()

# Проверяем количество записей
cursor.execute("SELECT COUNT(*) FROM auth_events")
count = cursor.fetchone()[0]
print(f"Записей в таблице: {count:,}")

In [ ]:
# Смотрим структуру таблицы
cursor.execute("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_name = 'auth_events'
    ORDER BY ordinal_position
""")

print("Структура таблицы auth_events:")
print("-" * 50)
for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<25} {'NULL' if row[2] == 'YES' else 'NOT NULL'}")

In [ ]:
# Смотрим примеры данных
cursor.execute("SELECT * FROM auth_events ORDER BY timestamp DESC LIMIT 5")
columns = [desc[0] for desc in cursor.description]

print("Примеры записей:")
for row in cursor.fetchall():
    print("-" * 50)
    for col, val in zip(columns, row):
        print(f"{col}: {val}")

In [ ]:
# Проверяем существующие индексы
cursor.execute("""
    SELECT indexname, indexdef
    FROM pg_indexes
    WHERE tablename = 'auth_events'
""")

print("Существующие индексы:")
for row in cursor.fetchall():
    print(f"\n{row[0]}:")
    print(f"  {row[1]}")

## Часть 2: EXPLAIN ANALYZE — профилирование запросов

EXPLAIN ANALYZE показывает, как PostgreSQL выполняет запрос и сколько времени занимает каждая операция.

In [ ]:
def explain_analyze(cursor, query):
    """Выполняет EXPLAIN ANALYZE и выводит результат."""
    cursor.execute(f"EXPLAIN ANALYZE {query}")
    print("План выполнения:")
    print("=" * 80)
    for row in cursor.fetchall():
        print(row[0])
    print("=" * 80)

In [ ]:
# Запрос 1: TOP-10 IP по неудачным попыткам входа
query1 = """
SELECT source_ip, COUNT(*) as failed_count
FROM auth_events
WHERE success = false
  AND timestamp > NOW() - INTERVAL '24 hours'
GROUP BY source_ip
ORDER BY failed_count DESC
LIMIT 10
"""

print("Запрос: TOP-10 IP по неудачным попыткам входа")
explain_analyze(cursor, query1)

In [ ]:
# Запрос 2: Распределение событий по часам суток
query2 = """
SELECT EXTRACT(HOUR FROM timestamp) as hour, COUNT(*) as event_count
FROM auth_events
GROUP BY EXTRACT(HOUR FROM timestamp)
ORDER BY hour
"""

print("Запрос: Распределение событий по часам")
explain_analyze(cursor, query2)

In [ ]:
# Запрос 3: Потенциальный brute-force (успех после серии неудач)
query3 = """
WITH user_attempts AS (
    SELECT 
        username,
        source_ip,
        timestamp,
        success,
        LAG(success) OVER (PARTITION BY username, source_ip ORDER BY timestamp) as prev_success
    FROM auth_events
    WHERE timestamp > NOW() - INTERVAL '24 hours'
)
SELECT username, source_ip, COUNT(*) as suspicious_count
FROM user_attempts
WHERE success = true AND prev_success = false
GROUP BY username, source_ip
HAVING COUNT(*) > 1
ORDER BY suspicious_count DESC
LIMIT 10
"""

print("Запрос: Потенциальный brute-force")
explain_analyze(cursor, query3)

### Создание индексов

На основе анализа запросов создадим индексы для ускорения.

In [ ]:
# Создаём индексы (если их ещё нет)
indexes = [
    # Индекс для фильтрации по времени
    "CREATE INDEX IF NOT EXISTS idx_auth_timestamp ON auth_events(timestamp)",
    
    # Индекс для фильтрации по успешности
    "CREATE INDEX IF NOT EXISTS idx_auth_success ON auth_events(success)",
    
    # Составной индекс для частого паттерна
    "CREATE INDEX IF NOT EXISTS idx_auth_success_timestamp ON auth_events(success, timestamp)",
    
    # Индекс для группировки по IP
    "CREATE INDEX IF NOT EXISTS idx_auth_source_ip ON auth_events(source_ip)",
]

for idx_sql in indexes:
    print(f"Выполняю: {idx_sql[:60]}...")
    cursor.execute(idx_sql)

conn.commit()
print("\nИндексы созданы!")

In [ ]:
# Обновляем статистику для оптимизатора
cursor.execute("ANALYZE auth_events")
conn.commit()
print("Статистика обновлена")

In [ ]:
# Повторяем запрос 1 после создания индексов
print("Запрос после создания индексов:")
explain_analyze(cursor, query1)

### Замер времени выполнения

In [ ]:
def measure_query(cursor, query, name):
    """Измеряет время выполнения запроса."""
    start = time.perf_counter()
    cursor.execute(query)
    results = cursor.fetchall()
    elapsed = time.perf_counter() - start
    print(f"{name}: {elapsed*1000:.2f} мс ({len(results)} строк)")
    return elapsed, results

print("PostgreSQL — время выполнения:")
print("-" * 50)
pg_times = {}
pg_times['query1'], _ = measure_query(cursor, query1, "TOP-10 IP")
pg_times['query2'], _ = measure_query(cursor, query2, "По часам")
pg_times['query3'], _ = measure_query(cursor, query3, "Brute-force")

## Часть 3: DuckDB — знакомство

DuckDB можно использовать как встроенную базу данных прямо в Python. Нулевая настройка!

In [ ]:
# DuckDB работает без настройки!
duckdb.sql("SELECT 'Hello, DuckDB!' as greeting").show()

# Демонстрация синтаксического сахара
print("\nFROM первым + GROUP BY ALL:")
duckdb.sql("""
    FROM (VALUES 
        ('192.168.1.1', 'blocked'),
        ('10.0.0.5', 'allowed'),
        ('192.168.1.1', 'blocked')
    ) AS t(ip, status)
    SELECT ip, status, COUNT(*) as cnt
    GROUP BY ALL
""").show()

## Часть 4: DuckDB + PostgreSQL

DuckDB может читать данные напрямую из PostgreSQL.

In [ ]:
# Устанавливаем и загружаем расширение postgres
duckdb.sql("INSTALL postgres; LOAD postgres;")

# Подключаемся к PostgreSQL
duckdb.sql(f"ATTACH '{PG_CONNECTION_STRING}' AS pg (TYPE postgres, READ_ONLY)")
print("Подключено к PostgreSQL")

# Проверяем таблицы
duckdb.sql("SHOW TABLES FROM pg").show()

In [ ]:
# Замеряем время выполнения через DuckDB → PostgreSQL
def measure_duckdb(query, name):
    start = time.perf_counter()
    result = duckdb.sql(query).fetchall()
    elapsed = time.perf_counter() - start
    print(f"{name}: {elapsed*1000:.2f} мс ({len(result)} строк)")
    return elapsed

print("DuckDB → PostgreSQL — время выполнения:")
print("-" * 50)
duckdb_pg_times = {}

duckdb_pg_times['query1'] = measure_duckdb("""
    FROM pg.auth_events
    SELECT source_ip, COUNT(*) as failed_count
    WHERE success = false AND timestamp > NOW() - INTERVAL '24 hours'
    GROUP BY ALL
    ORDER BY failed_count DESC
    LIMIT 10
""", "TOP-10 IP")

duckdb_pg_times['query2'] = measure_duckdb("""
    FROM pg.auth_events
    SELECT EXTRACT(HOUR FROM timestamp) as hour, COUNT(*) as event_count
    GROUP BY ALL
    ORDER BY hour
""", "По часам")

## Часть 5: Экспорт в Parquet и сравнение

In [ ]:
# Экспортируем данные из PostgreSQL в Parquet
import os

print("Экспорт в Parquet...")
start = time.perf_counter()
duckdb.sql("""
    COPY (SELECT * FROM pg.auth_events) 
    TO 'data/auth_events.parquet' (FORMAT PARQUET)
""")
elapsed = time.perf_counter() - start
print(f"Экспорт завершён за {elapsed:.2f} сек")

# Размер файла
parquet_size = os.path.getsize("data/auth_events.parquet")
print(f"Размер Parquet: {parquet_size / 1024:.1f} KB")

In [ ]:
# Запросы по Parquet
print("DuckDB → Parquet — время выполнения:")
print("-" * 50)
duckdb_parquet_times = {}

duckdb_parquet_times['query1'] = measure_duckdb("""
    FROM 'data/auth_events.parquet'
    SELECT source_ip, COUNT(*) as failed_count
    WHERE success = false AND timestamp > NOW() - INTERVAL '24 hours'
    GROUP BY ALL
    ORDER BY failed_count DESC
    LIMIT 10
""", "TOP-10 IP")

duckdb_parquet_times['query2'] = measure_duckdb("""
    FROM 'data/auth_events.parquet'
    SELECT EXTRACT(HOUR FROM timestamp) as hour, COUNT(*) as event_count
    GROUP BY ALL
    ORDER BY hour
""", "По часам")

In [ ]:
# Сводная таблица сравнения
print("\n" + "=" * 65)
print("СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ (мс)")
print("=" * 65)
print(f"{'Запрос':<15} {'PostgreSQL':>15} {'DuckDB→PG':>15} {'DuckDB→Parquet':>15}")
print("-" * 65)

for qname, label in [('query1', 'TOP-10 IP'), ('query2', 'По часам')]:
    pg = pg_times.get(qname, 0) * 1000
    dpg = duckdb_pg_times.get(qname, 0) * 1000
    dpar = duckdb_parquet_times.get(qname, 0) * 1000
    print(f"{label:<15} {pg:>15.2f} {dpg:>15.2f} {dpar:>15.2f}")

print("=" * 65)

In [ ]:
# Закрываем соединения
cursor.close()
conn.close()
duckdb.sql("DETACH pg")
print("Соединения закрыты")

## Выводы

**PostgreSQL** хорош для:
- Записи новых событий в реальном времени
- Поиска конкретных записей по индексам
- Транзакционной целостности

**DuckDB** хорош для:
- Аналитических запросов с агрегациями
- Работы с файлами (CSV, JSON, Parquet)
- Ad-hoc анализа без настройки инфраструктуры

**Parquet** — оптимальный формат для:
- Архивного хранения логов
- Передачи данных между системами
- Аналитики больших объёмов

**Рекомендуемый workflow:**
1. Vector → PostgreSQL (свежие данные)
2. PostgreSQL → Parquet (архив)
3. DuckDB → Parquet (аналитика)